In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [2]:
def calculate_correlation_between_dfs(df1, df2, start_date=None, end_date=None, method='pearson', min_periods=4):
    """
    두 개의 시계열 DataFrame의 상관관계를 계산하되, 유효 관측치가 min_periods보다 많을 경우만 수행

    Parameters:
    ...
    - min_periods (int): 최소 유효 데이터 수

    Returns:
    - pd.DataFrame: 상관계수 매트릭스
    """
    if start_date:
        df1 = df1[df1.index >= pd.to_datetime(start_date)]
        df2 = df2[df2.index >= pd.to_datetime(start_date)]
    if end_date:
        df1 = df1[df1.index <= pd.to_datetime(end_date)]
        df2 = df2[df2.index <= pd.to_datetime(end_date)]

    combined = pd.merge(df1, df2, left_index=True, right_index=True, how='inner', suffixes=('_firm', '_hs'))

    corr_matrix = pd.DataFrame(index=df1.columns, columns=df2.columns, dtype=float)

    for firm in df1.columns:
        for hs in df2.columns:
            x = combined[firm]
            y = combined[hs]
            valid = x.notna() & y.notna()
            if valid.sum() >= min_periods:
                corr_matrix.loc[firm, hs] = x[valid].corr(y[valid], method=method)
            else:
                corr_matrix.loc[firm, hs] = np.nan  # 또는 0

    return corr_matrix

def get_top_correlated_hscode(corr_matrix, symbol, top_n=5, threshold=None, ascending=False):
    """
    특정 기업(Symbol)에 대해 상관관계가 높은 HS 코드를 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - symbol (str): 대상 Symbol (예: '000080')
    - top_n (int): 상위 N개 추출 (threshold와 함께 사용 시 무시될 수 있음)
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기). 설정 시 top_n보다 우선함
    - ascending (bool): 상관계수 기준 오름차순 정렬 여부 (기본값: False = 높은 값 우선)

    Returns:
    - pd.DataFrame: root_hs_code 및 상관계수를 포함한 상위 N개 HS 코드
    """

    if symbol not in corr_matrix.index:
        raise ValueError(f"Symbol '{symbol}' not found in correlation matrix.")

    symbol_corr = corr_matrix.loc[symbol].dropna()

    if threshold is not None:
        symbol_corr = symbol_corr[symbol_corr >= threshold]

    top_correlated = symbol_corr.sort_values(ascending=ascending).head(top_n)

    return top_correlated.reset_index().rename(columns={'index': 'root_hs_code', symbol: 'correlation'})

def get_top_correlated_symbols(corr_matrix, hs_code, top_n=5, threshold=None, ascending=False):
    """
    특정 HS 코드에 대해 상관관계가 높은 기업 Symbol을 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - hs_code (str or int): 대상 HS 코드 (예: '151550')
    - top_n (int): 상위 N개 추출
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기)
    - ascending (bool): 정렬 방향 (False: 높은 상관 우선)

    Returns:
    - pd.DataFrame: symbol 및 correlation 정보를 담은 상위 N개 결과
    """

    if hs_code not in corr_matrix.columns:
        raise ValueError(f"HS code '{hs_code}' not found in correlation matrix columns.")

    hs_corr = corr_matrix[hs_code].dropna()

    if threshold is not None:
        hs_corr = hs_corr[hs_corr >= threshold]

    top_symbols = hs_corr.sort_values(ascending=ascending).head(top_n)

    return top_symbols.reset_index().rename(columns={'index': 'symbol', hs_code: 'correlation'})


In [3]:
db_info = {
    'host': 'hystox74.synology.me',
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 테이블 이름
table_name = 'target_hs_code'

# 고유한 hs_code 값 추출 쿼리 실행
query = f"SELECT DISTINCT hs_code FROM {table_name}"
unique_hs_codes_df = pd.read_sql(query, con=engine)
hs_codes  = unique_hs_codes_df['hs_code'].unique().tolist()

indicator = 'expDlr'

df_real = fetch_trade_data_multi_hscode(db_info, hs_codes, indicator)

# 분기 정보 추가
df_real['quarter'] = df_real['date'].dt.to_period('Q')

# 그룹별로 분기별 합산
df_quarterly = (
    df_real
    .groupby(['root_hs_code', 'quarter'])['value']
    .sum()
    .reset_index()
)

# 👉 분기 월말로 변환 (예: 2007Q1 → 2007-03-31)
df_quarterly['date'] = df_quarterly['quarter'].dt.to_timestamp(how='end')

# 👉 'quarter' 컬럼 제거
df_quarterly.drop(columns=['quarter'], inplace=True)

# 1단계: 문자열로 직접 변환하려면 to_datetime 이후에 바로 strftime
df_quarterly['date'] = pd.to_datetime(df_quarterly['date']).dt.strftime('%Y-%m-%d')

def create_yoy_growth_pivot(df_quarterly, start_date=None, end_date=None):
    """
    전년 동분기 대비 증가율을 pivot 형태로 변환하고 분석기간을 설정할 수 있는 함수

    Parameters:
    - df_quarterly (DataFrame): 'root_hs_code', 'date', 'yoy_growth' 포함된 데이터
    - start_date (str or None): 분석 시작일 (예: '2015-01-01')
    - end_date (str or None): 분석 종료일 (예: '2023-12-31')

    Returns:
    - pivot_df (DataFrame): 행: date, 열: root_hs_code, 값: yoy_growth
    """
    # Pivot
    pivot_df = df_quarterly.pivot(
        index='date',
        columns='root_hs_code',
        values='yoy_growth'
    ).sort_index()

    # inf 값 NaN 처리
    pivot_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 분석 기간 슬라이싱 (날짜가 문자열이면 datetime으로 변환)
    pivot_df.index = pd.to_datetime(pivot_df.index)

    if start_date:
        pivot_df = pivot_df[pivot_df.index >= pd.to_datetime(start_date)]
    if end_date:
        pivot_df = pivot_df[pivot_df.index <= pd.to_datetime(end_date)]

    return pivot_df


# 전년 동분기 값 (4개 분기 전 값) 계산
df_quarterly['yoy_value'] = (
    df_quarterly
    .sort_values(['root_hs_code', 'date'])
    .groupby('root_hs_code')['value']
    .shift(4)
)

# ❗ yoy_growth 계산
df_quarterly['yoy_growth'] = (
    (df_quarterly['value'] - df_quarterly['yoy_value']) / df_quarterly['yoy_value']
) * 100

quarterly_trade_data = create_yoy_growth_pivot(df_quarterly, start_date='2008-03', end_date='2025-03')

In [4]:
fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 1. indicator 필터링
target_indicator = '매출액(천원)'
filtered_df = fs_df[fs_df['indicator'] == target_indicator].copy()

# 2. 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)

# 3. value 컬럼이 있는지 확인 및 타입 강제
if 'value' not in filtered_df.columns:
    raise KeyError("'value' 컬럼이 없습니다.")

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')

# 4. 피벗 테이블 생성 (행: date, 열: Symbol, 값: value)
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='symbol',
    values='value',
    aggfunc='first'  # 중복 방지
)

# 5. 전년 동분기 대비 변화율 계산 (4분기 전 대비)
fs_yoy_growth_df = pivot_df.pct_change(periods=4) * 100

✅ 'korea_fs_data' 테이블에서 5584577건의 데이터를 가져왔습니다.


In [19]:
# correlation_result.to_csv(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수.csv", index=True, encoding="utf-8-sig")

path = r"C:\Users\MetaM\PycharmProjects\stock_forecast\DATA\한국상장사_수출데이터_상관계수2.csv"

correlation_result = pd.read_csv(path).set_index('symbol')

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\MetaM\\PycharmProjects\\stock_forecast\\DATA\\한국상장사_수출데이터_상관계수2.csv'

In [14]:
correlation_result

,121120,121221,151550,151590,170199,190230,190590,200599,200830,200899,...,903149,903180,903190,903289,940130,940199,940330,940540,950300,970191
symbol,,,,,,,,,,,,,,,,,,,,,
A000010,0.310617,0.044100,-0.256057,-0.189933,0.131040,-0.156207,-0.236201,-0.274586,0.031884,-0.224384,...,0.613469,-0.171279,-0.222212,0.034071,-0.278841,-0.853721,0.371479,-0.655051,-0.059277,0.732864
A000020,0.124735,0.225071,-0.068189,-0.085932,-0.229223,-0.015802,-0.169895,-0.331209,-0.064024,-0.355318,...,0.098013,-0.078190,-0.093215,-0.037474,-0.196819,0.429429,0.240497,-0.171796,0.312544,-0.086697
A000030,0.300940,0.095613,-0.230592,-0.141457,0.177613,-0.124090,-0.223985,-0.389654,0.006780,-0.152835,...,0.624748,-0.157941,-0.225218,-0.015740,-0.424519,-0.864607,0.316282,-0.728726,-0.073997,0.749627
A000040,0.001785,-0.181251,0.008394,0.063539,-0.423618,-0.342226,-0.184077,-0.058404,-0.117799,0.128385,...,-0.195238,-0.061765,0.062224,-0.156954,0.084563,-0.071856,0.193858,0.089512,-0.027562,0.252328
A000050,-0.239210,0.014402,0.049310,0.117870,0.156750,0.030490,0.306476,-0.113787,0.004677,-0.245099,...,-0.312176,0.352827,-0.067670,0.129579,-0.193828,0.376524,0.302681,0.092565,0.200296,-0.532924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A950180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.578113,NaN,NaN,NaN,0.822625
A950190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.319711,NaN,NaN,NaN,0.085886
A950200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.548365,NaN,NaN,NaN,-0.235305


In [17]:
top_hs_codes = get_top_correlated_hscode(
    corr_matrix=correlation_result,  # 이전에 만든 상관관계 행렬
    symbol='A257720',
    top_n=50,
    threshold=0.2  # 선택사항
)

print(top_hs_codes)

   root_hs_code  correlation
0        851419     0.835153
1        852491     0.827986
2        852411     0.782651
3        852412     0.736177
4        880730     0.731744
5        854142     0.708228
6        854141     0.557047
7        854149     0.422579
8        852492     0.386001
9        852589     0.329177
10       940199     0.308667
11       741980     0.299101
12       846262     0.275094
13       852499     0.212281
14       851779     0.207472


In [18]:
top_symbols = get_top_correlated_symbols(
    corr_matrix=correlation_result,
    hs_code='330499',
    top_n=50,
    threshold=0.1  # 선택사항
)

print(top_symbols)

     symbol  correlation
0   A074130     0.967511
1   A192820     0.729322
2   A016100     0.728912
3   A035720     0.717058
4   A053740     0.708166
5   A069110     0.701793
6   A003470     0.686604
7   A002790     0.682041
8   A090430     0.681187
9   A036490     0.680674
10  A051900     0.651609
11  A005750     0.632255
12  A068150     0.623032
13  A000540     0.608105
14  A078340     0.600840
15  A036210     0.578163
16  A950100     0.578163
17  A053000     0.578163
18  A044820     0.573625
19  A114570     0.569294
20  A104120     0.568078
21  A009240     0.563822
22  A052220     0.559220
23  A030190     0.534288
24  A031440     0.524754
25  A027390     0.522229
26  A060540     0.518554
27  A014820     0.514147
28  A059100     0.510036
29  A072950     0.509532
30  A090080     0.508573
31  A000370     0.499764
32  A272290     0.498847
33  A078940     0.495821
34  A344820     0.494364
35  A086390     0.491161
36  A063170     0.489228
37  A104040     0.488845
38  A302440     0.485732
